In [ ]:
# %pip install ninja ipykernel ipywidgets huggingface_hub --break-system-packages

In [ ]:
# %pip install --upgrade transformers --break-system-packages

In [ ]:
# %pip install git+https://github.com/intel/auto-round.git --break-system-packages

In [ ]:
# %pip install git+https://github.com/sustcsonglin/flash-linear-attention.git --no-build-isolation --break-system-packages

In [ ]:
# %pip install git+https://github.com/Dao-AILab/causal-conv1d.git --no-build-isolation --break-system-packages

In [ ]:
# %pip install compressed-tensors --break-system-packages

In [ ]:
# %pip install tilelang --break-system-packages

In [1]:
import os

import torch
from auto_round import AutoRound, AutoScheme
from huggingface_hub import HfApi, create_repo, get_token, notebook_login
from safetensors import safe_open
from transformers import AutoModelForImageTextToText, AutoProcessor


In [2]:
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [3]:
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA Version: {torch.version.cuda}")
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")


PyTorch Version: 2.12.0+cu130
CUDA Available: True
CUDA Version: 13.0
GPU Name: NVIDIA L40S
VRAM: 44.4 GB


In [4]:
notebook_login()

In [5]:
MODEL_ID = "Qwen/Qwen3.5-4B"
HF_USER = "Vishva007"
OUTPUT_BASE_DIR = "./AutoRound"
LOCAL_PATH = "./local_model"

In [6]:
!hf download $MODEL_ID --local-dir $LOCAL_PATH

Fetching 14 files: 100%|██████████████████████| 14/14 [00:00<00:00, 675.05it/s]
Download complete: : 0.00B [00:00, ?B/s]              ✓ Downloaded
  path: /workspace/local_model
Download complete: : 0.00B [00:00, ?B/s]


In [7]:
for file in os.listdir(LOCAL_PATH):
    if file.endswith(".safetensors"):
        path = os.path.join(LOCAL_PATH, file)
        print(f"\nChecking {file}")

        with safe_open(path, framework="pt") as f:
            keys = list(f.keys())

            mtp_keys = [k for k in keys if "mtp" in k.lower()]
            for k in mtp_keys:
                print(k)


Checking model.safetensors-00002-of-00002.safetensors
mtp.fc.weight
mtp.layers.0.input_layernorm.weight
mtp.layers.0.post_attention_layernorm.weight
mtp.layers.0.self_attn.k_norm.weight
mtp.layers.0.self_attn.k_proj.weight
mtp.layers.0.self_attn.o_proj.weight
mtp.layers.0.self_attn.q_norm.weight
mtp.layers.0.self_attn.q_proj.weight
mtp.layers.0.self_attn.v_proj.weight
mtp.norm.weight
mtp.pre_fc_norm_embedding.weight
mtp.pre_fc_norm_hidden.weight

Checking model.safetensors-00001-of-00002.safetensors
mtp.layers.0.mlp.down_proj.weight
mtp.layers.0.mlp.gate_proj.weight
mtp.layers.0.mlp.up_proj.weight


In [8]:
model = AutoModelForImageTextToText.from_pretrained(
    LOCAL_PATH, 
    dtype=torch.bfloat16, 
    device_map="auto"
)
processor = AutoProcessor.from_pretrained(LOCAL_PATH)

tokenizer = processor.tokenizer


Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

[ERROR] `min_frames` is part of Qwen3VLVideoProcessorInitKwargs, but not documented. Make sure to add it to the docstring of the function in /usr/local/lib/python3.12/dist-packages/transformers/models/qwen3_vl/video_processing_qwen3_vl.py.
[ERROR] `max_frames` is part of Qwen3VLVideoProcessorInitKwargs, but not documented. Make sure to add it to the docstring of the function in /usr/local/lib/python3.12/dist-packages/transformers/models/qwen3_vl/video_processing_qwen3_vl.py.


In [9]:
model

Qwen3_5ForConditionalGeneration(
  (model): Qwen3_5Model(
    (visual): Qwen3_5VisionModel(
      (patch_embed): Qwen3_5VisionPatchEmbed(
        (proj): Conv3d(3, 1024, kernel_size=(2, 16, 16), stride=(2, 16, 16))
      )
      (pos_embed): Embedding(2304, 1024)
      (rotary_pos_emb): Qwen3_5VisionRotaryEmbedding()
      (blocks): ModuleList(
        (0-23): 24 x Qwen3_5VisionBlock(
          (norm1): LayerNorm((1024,), eps=1e-06, elementwise_affine=True, bias=True)
          (norm2): LayerNorm((1024,), eps=1e-06, elementwise_affine=True, bias=True)
          (attn): Qwen3_5VisionAttention(
            (qkv): Linear(in_features=1024, out_features=3072, bias=True)
            (proj): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (mlp): Qwen3_5VisionMLP(
            (linear_fc1): Linear(in_features=1024, out_features=4096, bias=True)
            (linear_fc2): Linear(in_features=4096, out_features=1024, bias=True)
            (act_fn): GELUTanh()
         

In [10]:
def push_to_hub(local_dir, repo_name, token):
    """Creates repo and uploads folder to Hugging Face."""
    full_repo_id = f"{HF_USER}/{repo_name}"
    print(f"\n[Hub] Pushing {local_dir} to {full_repo_id}...")

    try:
        api = HfApi()
        create_repo(
            full_repo_id, repo_type="model", exist_ok=True, private=False, token=token
        )

        api.upload_folder(
            folder_path=local_dir, repo_id=full_repo_id, repo_type="model", token=token
        )
        print(f"[Hub] ✅ Successfully uploaded: https://huggingface.co/{full_repo_id}")
    except Exception as e:
        print(f"[Hub] ❌ Error uploading: {e}")

In [11]:
# Exclude low-rank state gates, lm_head, and speculative layers
TUNING_CONFIG = {
    "group_size": 128,              # Standardize to 128 for tensor alignment
    "sym": True,
    "iters": 10,                   # Stable accuracy calibration
    "nsamples": 512,
    "batch_size": 8,                # Lower batch size prevents memory pressure
    "seqlen": 2048,
    "low_gpu_mem_usage": True,
    "enable_torch_compile": True,
    "quant_nontext_module": False,   # Keeps visual patch embed & blocks in BF16
    "layer_config": {
        # Recurrent state projections (out_features=32) must remain BF16
        "*.linear_attn.in_proj_a": {"data_type": "bfloat16"},
        "*.linear_attn.in_proj_b": {"data_type": "bfloat16"},
        # Output prediction head
        "lm_head": {"data_type": "bfloat16"},
        # Speculative draft components
        "mtp": {"data_type": "bfloat16"},
        "mtp.fc": {"data_type": "bfloat16"}
    }
}

In [12]:
shared_layers = [
    # Attention QKV projections
    ["q_proj", "k_proj", "v_proj"],
    # Linear attention fused input projections (excluding decay/gates a & b)
    ["in_proj_qkv", "in_proj_z"],
    # SwiGLU / MLP Gate & Up projections
    ["gate_proj", "up_proj"],
]

In [13]:
mixed_scheme = AutoScheme(
    avg_bits=3.5,                       # The target average bit-width for the entire model
    options=["W4A16", "W3A16"],         # The allowed bit-widths to scale between
    shared_layers=shared_layers,
    enable_torch_compile=True,
)


In [14]:

ar = AutoRound(
    model=model,
    tokenizer=tokenizer,
    processor=processor,
    scheme=mixed_scheme,
    **TUNING_CONFIG,
)

2026-08-17 17:02:55 WARNING autoround.py L554: Passing 'group_size' directly to AutoRound is supported, but the recommended usage is 'alg_configs=SignRoundConfig(...)'.
2026-08-17 17:02:55 WARNING autoround.py L554: Passing 'sym' directly to AutoRound is supported, but the recommended usage is 'alg_configs=SignRoundConfig(...)'.
2026-08-17 17:02:55 WARNING autoround.py L554: Passing 'iters' directly to AutoRound is supported, but the recommended usage is 'alg_configs=SignRoundConfig(...)'.


In [15]:
ar.quantize_and_save(
    OUTPUT_BASE_DIR, format="llm_compressor", inplace=True
)

[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.
2026-08-17 17:02:55 INFO base.py L717: AutoScheme on multimodal LLM: scoring the language tower only with text-only calibration (multimodal dataloader will be used as a fallback if needed).
2026-08-17 17:02:55 WARNING logging.py L340: Layer name or regex '*.linear_attn.in_proj_a' in layer_config does not match any supported layers. Please check for typos or update the regex pattern, ignore it for now
2026-08-17 17:02:55 WARNING logging.py L340: Layer name or regex '*.linear_attn.in_proj_b' in layer_config does not match any supported layers. Please check for typos or update the regex pattern, ignore it for now
2026-08-17 17:02:56 WARNING resolver.py L60: reset `quant_lm_head` to false as quantizing lm_head with tied weights has not been supported currently
2026-08-17 17:02:56 INFO gen_auto_scheme.py L266: AutoScheme option QuantizationScheme(bits=4, group_size=128, s

(Qwen3_5ForConditionalGeneration(
   (model): Qwen3_5Model(
     (visual): Qwen3_5VisionModel(
       (patch_embed): Qwen3_5VisionPatchEmbed(
         (proj): Conv3d(3, 1024, kernel_size=(2, 16, 16), stride=(2, 16, 16))
       )
       (pos_embed): Embedding(2304, 1024)
       (rotary_pos_emb): Qwen3_5VisionRotaryEmbedding()
       (blocks): ModuleList(
         (0-23): 24 x Qwen3_5VisionBlock(
           (norm1): LayerNorm((1024,), eps=1e-06, elementwise_affine=True, bias=True)
           (norm2): LayerNorm((1024,), eps=1e-06, elementwise_affine=True, bias=True)
           (attn): Qwen3_5VisionAttention(
             (qkv): Linear(in_features=1024, out_features=3072, bias=True)
             (proj): Linear(in_features=1024, out_features=1024, bias=True)
           )
           (mlp): Qwen3_5VisionMLP(
             (linear_fc1): Linear(in_features=1024, out_features=4096, bias=True)
             (linear_fc2): Linear(in_features=4096, out_features=1024, bias=True)
             (act_fn): 

In [16]:
base_name = MODEL_ID.split("/")[-1]
hf_token = get_token()

In [17]:
if hf_token:
    push_to_hub(
        os.path.join(OUTPUT_BASE_DIR, "local_model-w4g128"), 
        f"{base_name}-Mixed-3.5bit-AutoRound",
        hf_token
    )
else:
    print("No Hugging Face token found. Skipping upload to hub.")


[Hub] Pushing ./AutoRound/local_model-w4g128 to Vishva007/Qwen3.5-4B-Mixed-3.5bit-AutoRound...


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

[Hub] ✅ Successfully uploaded: https://huggingface.co/Vishva007/Qwen3.5-4B-Mixed-3.5bit-AutoRound
